## 1. Setup

### Class Imbalance Strategy and Evaluation Choice

With approximately **8-9% positive rate**, the `hospital_death` target is moderately
imbalanced. This matters for both model fitting and evaluation:

**Imbalance handling:**
- Logistic Regression: use `class_weight='balanced'` to up-weight minority class.
- XGBoost / LightGBM: use `scale_pos_weight` = (n_negative / n_positive) to
  penalise false negatives more heavily during gradient computation.

**Evaluation choice — AUC-PR over AUC-ROC:**
- ROC-AUC is insensitive to class imbalance because it averages over all
  classification thresholds including many that would never be clinically used.
  A model predicting all patients as survivors can still achieve ROC-AUC > 0.5.
- **Precision-Recall AUC** (average precision) focuses on the minority class
  and directly reflects the trade-off between catching true deaths (recall)
  and avoiding false alarms (precision) — the clinically relevant trade-off.
- **Brier Score** measures calibration — how close predicted probabilities are
  to true outcomes — essential for threshold selection in deployment.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import split_data, scale_numeric
from src.models import get_model, train, evaluate, save_model

os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Load engineered features
df = pd.read_csv('../data/processed/features_engineered.csv')
print(f'Loaded dataset: {df.shape}')
print(f'Target rate: {df["hospital_death"].mean():.4f}')

# Split into train/val/test
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df, target='hospital_death')
print(f'\nSplit sizes — Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Train positive rate: {y_train.mean():.4f}')

# Scale numeric features
X_train_s, X_val_s, X_test_s, scaler = scale_numeric(X_train, X_val, X_test)
print(f'\nScaling applied. Train shape: {X_train_s.shape}')

ModuleNotFoundError: No module named 'lightgbm'

## 2. Logistic Regression Baseline

Logistic Regression serves as a **strong linear baseline**. Despite its simplicity,
a well-regularised LR model often achieves surprisingly competitive performance on
tabular data. It also produces well-calibrated probabilities, making the Brier score
a meaningful comparison point against tree-based models.

We use `class_weight='balanced'` to account for class imbalance and L2 regularisation
(default). The validation metrics establish the floor that more complex models must beat.

In [ ]:
results = {}

print('=== Logistic Regression Baseline ===')
lr_model = get_model('lr')
lr_model = train(lr_model, X_train_s, y_train)
lr_metrics = evaluate(lr_model, X_val_s, y_val)

results['Logistic Regression'] = lr_metrics

print(f"  ROC-AUC:  {lr_metrics['roc_auc']:.4f}")
print(f"  PR-AUC:   {lr_metrics['pr_auc']:.4f}")
print(f"  Brier:    {lr_metrics['brier']:.4f}")
if 'f1' in lr_metrics:
    print(f"  F1:       {lr_metrics['f1']:.4f}")

## 3. XGBoost

XGBoost is a gradient boosted tree ensemble that frequently tops tabular benchmarks.
Key hyperparameters:
- `scale_pos_weight`: compensates for class imbalance by weighting positive samples.
- `early_stopping_rounds=50`: prevents overfitting by halting when validation
  PR-AUC stops improving.
- `subsample` and `colsample_bytree`: stochastic tree building reduces variance.

We pass `y_train` to `get_model` so the function can compute the appropriate
`scale_pos_weight` ratio automatically.

In [ ]:
print('=== XGBoost ===')
xgb_model = get_model('xgb', y_train=y_train)
xgb_model = train(
    xgb_model, X_train_s, y_train,
    eval_set=[(X_val_s, y_val)],
    early_stopping_rounds=50,
    verbose=100
)
xgb_metrics = evaluate(xgb_model, X_val_s, y_val)

results['XGBoost'] = xgb_metrics

print(f"\n  ROC-AUC:  {xgb_metrics['roc_auc']:.4f}")
print(f"  PR-AUC:   {xgb_metrics['pr_auc']:.4f}")
print(f"  Brier:    {xgb_metrics['brier']:.4f}")
if 'f1' in xgb_metrics:
    print(f"  F1:       {xgb_metrics['f1']:.4f}")

## 4. LightGBM

LightGBM uses a **leaf-wise growth strategy** (vs. XGBoost's level-wise) making it
significantly faster on large datasets while often achieving equal or better accuracy.
Key advantages for this task:
- Native handling of `NaN` values (useful if any residual missingness remains).
- `is_unbalance=True` as an alternative to explicit `scale_pos_weight`.
- Faster per-iteration training allows exploring a larger number of estimators.

LightGBM is our **champion model candidate** based on prior benchmarks on similar
ICU datasets.

In [ ]:
print('=== LightGBM ===')
lgbm_model = get_model('lgbm', y_train=y_train)
lgbm_model = train(
    lgbm_model, X_train_s, y_train,
    eval_set=[(X_val_s, y_val)],
    early_stopping_rounds=50,
    verbose=100
)
lgbm_metrics = evaluate(lgbm_model, X_val_s, y_val)

results['LightGBM'] = lgbm_metrics

print(f"\n  ROC-AUC:  {lgbm_metrics['roc_auc']:.4f}")
print(f"  PR-AUC:   {lgbm_metrics['pr_auc']:.4f}")
print(f"  Brier:    {lgbm_metrics['brier']:.4f}")
if 'f1' in lgbm_metrics:
    print(f"  F1:       {lgbm_metrics['f1']:.4f}")

## 5. Model Comparison

We compare all three models on the **validation set** across three complementary metrics:
- **ROC-AUC**: discrimination across all thresholds.
- **PR-AUC**: precision-recall trade-off, more relevant under imbalance.
- **Brier Score**: probabilistic calibration (lower is better).

The model with the best PR-AUC is selected as the champion for test set evaluation.

In [ ]:
metrics_df = pd.DataFrame(results).T[['roc_auc', 'pr_auc', 'brier']].round(4)
print('Validation Set Metrics:')
print(metrics_df.to_string())

# Champion model selection
champion_name = metrics_df['pr_auc'].idxmax()
print(f'\nChampion model (best PR-AUC): {champion_name}')

# Grouped bar chart
fig, ax = plt.subplots(figsize=(11, 6))
metric_names = ['roc_auc', 'pr_auc', 'brier']
metric_labels = ['ROC-AUC', 'PR-AUC', 'Brier Score']
model_names = metrics_df.index.tolist()
n_models = len(model_names)
n_metrics = len(metric_names)
bar_width = 0.22
x = np.arange(n_metrics)
colors = ['#4878CF', '#6ACC65', '#D65F5F']

for i, (model, color) in enumerate(zip(model_names, colors)):
    vals = [metrics_df.loc[model, m] for m in metric_names]
    offset = (i - n_models / 2 + 0.5) * bar_width
    bars = ax.bar(x + offset, vals, bar_width, label=model, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylabel('Metric Value')
ax.set_title('Model Comparison — Validation Set Metrics', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', bbox_inches='tight')
plt.show()

## 6. Test Set Evaluation (Champion Model)

We evaluate the champion model on the **held-out test set** — data the model has
never seen in any form during training or hyperparameter selection. This provides
the unbiased performance estimate for reporting. The model is then serialised to
disk for use in explainability, fairness audit, and the LangChain Q&A interface.

In [ ]:
# Map champion name to model object
model_registry = {
    'Logistic Regression': lr_model,
    'XGBoost': xgb_model,
    'LightGBM': lgbm_model
}
champion_model = model_registry[champion_name]

test_metrics = evaluate(champion_model, X_test_s, y_test)

print(f'=== Test Set Results: {champion_name} ===')
print(f"  ROC-AUC:  {test_metrics['roc_auc']:.4f}")
print(f"  PR-AUC:   {test_metrics['pr_auc']:.4f}")
print(f"  Brier:    {test_metrics['brier']:.4f}")

# Compare validation vs test (check for overfitting)
val_metrics = results[champion_name]
print(f'\nValidation vs Test gap:')
for m in ['roc_auc', 'pr_auc', 'brier']:
    delta = test_metrics[m] - val_metrics[m]
    print(f"  {m}: val={val_metrics[m]:.4f}, test={test_metrics[m]:.4f}, delta={delta:+.4f}")

save_model(champion_model, 'lgbm_best')
print('\nChampion model saved as lgbm_best')

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

y_pred_proba = champion_model.predict_proba(X_test_s)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc_val = auc(fpr, tpr)

precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
pr_auc_val = auc(recall, precision)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# ROC Curve
axes[0].plot(fpr, tpr, color='#4878CF', lw=2.5, label=f'ROC (AUC = {roc_auc_val:.4f})')
axes[0].plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1.2, label='Random classifier')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#4878CF')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[0].set_title(f'ROC Curve — {champion_name} (Test Set)', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=11)
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.02])

# Precision-Recall Curve
axes[1].plot(recall, precision, color='#D65F5F', lw=2.5, label=f'PR (AUC = {pr_auc_val:.4f})')
baseline = y_test.mean()
axes[1].axhline(baseline, color='grey', linestyle='--', lw=1.2, label=f'No-skill baseline ({baseline:.3f})')
axes[1].fill_between(recall, precision, alpha=0.08, color='#D65F5F')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title(f'Precision-Recall Curve — {champion_name} (Test Set)', fontweight='bold')
axes[1].legend(loc='upper right', fontsize=11)
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.02])

plt.suptitle(f'Champion Model Performance — {champion_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/roc_pr_curves.png', bbox_inches='tight')
plt.show()
print(f'Test ROC-AUC: {roc_auc_val:.4f} | Test PR-AUC: {pr_auc_val:.4f}')